# ECG Arrhythmia Detection & Classification Using Hidden Markov Models (HMM)

**Database:** MIT-BIH Arrhythmia Database (PhysioNet)  
**Standard:** ANSI/AAMI EC57:1998  
**Architecture:** Class-Conditional Continuous Gaussian HMMs vs. Machine Learning Baselines

---

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ALL_48_RECORDS, AAMI_CLASSES, SAMPLING_RATE, RAW_DATA_DIR, FIGURES_DIR, PATIENT_SPLIT
from src.data_loader import load_record, generate_dataset_summary
from src.preprocessing import preprocess_ecg
from src.peak_detection import pan_tompkins_qrs_detector, evaluate_r_peak_detection
from src.segmentation import segment_record_data
from src.feature_extraction import extract_sequential_features, extract_tabular_features
from src.hmm_model import ClassConditionalHMM
from src.baseline_models import BaselineClassifiers
from src.evaluation import compute_aami_metrics, format_metrics_table, compare_models
from src.visualization import (
    plot_raw_vs_filtered,
    plot_r_peak_detection,
    plot_heartbeat_gallery,
    plot_confusion_matrix,
    plot_model_comparison_bars,
    plot_hmm_viterbi_states
)

print("Modules successfully imported. Total available recordings:", len(ALL_48_RECORDS))

## 1. Load an ECG Record (Record 100)

In [ ]:
data_100 = load_record("100", data_dir=RAW_DATA_DIR)
print(f"Record ID:       {data_100['record_id']}")
print(f"Sampling Rate:   {data_100['fs']} Hz")
print(f"Channels:        {data_100['lead_names']}")
print(f"Signal Shape:    {data_100['signal'].shape}")
print(f"Duration:        {data_100['duration_sec'] / 60.0:.1f} minutes")
print(f"Total Annotated Beats: {len(data_100['ann_samples']):,}")

## 2. Signal Preprocessing & Filtering
- 0.5 - 45 Hz Zero-Phase Butterworth Bandpass Filter (filtfilt)
- 60 Hz Notch Filter
- Z-Score Normalization

In [ ]:
raw_sig = data_100["signal"][:, 0]
filtered_sig = preprocess_ecg(raw_sig, fs=SAMPLING_RATE, norm_method="zscore")

fig = plot_raw_vs_filtered(raw_sig, filtered_sig, fs=SAMPLING_RATE, duration_sec=10.0, start_sec=10.0, record_id="100")
plt.show()

## 3. Pan-Tompkins QRS Peak Detection
Compare detected peaks with gold-standard cardiologist annotations within $\pm 50\text{ ms}$ tolerance.

In [ ]:
detected_peaks = pan_tompkins_qrs_detector(raw_sig, fs=SAMPLING_RATE)
ref_peaks = data_100["ann_samples"]

eval_res = evaluate_r_peak_detection(detected_peaks, ref_peaks)
for k, v in eval_res.items():
    print(f"{k:>20}: {v}")

fig = plot_r_peak_detection(raw_sig, detected_peaks, ref_peaks, fs=SAMPLING_RATE, duration_sec=8.0, start_sec=12.0, record_id="100")
plt.show()

## 4. Heartbeat Segmentation & AAMI 5-Class Categorization

In [ ]:
seg_100 = segment_record_data(data_100, preprocessed_signal=filtered_sig)
print("Segments shape:", seg_100["segments"].shape)
print("Labels count:   ", pd.Series(seg_100["aami_labels"]).value_counts().to_dict())

## 5. Sequential Feature Extraction & HMM Model Training
Train a Class-Conditional Continuous Gaussian HMM where each hidden state captures a cardiac cycle phase.

In [ ]:
X_seq, lengths = extract_sequential_features(
    seg_100["segments"],
    seg_100["rr_pre"],
    seg_100["rr_post"],
    seg_100["rr_ratio"],
    n_frames=11
)
print("Sequential HMM observation matrix:", X_seq.shape)

# Train HMM
hmm = ClassConditionalHMM(n_states=5, n_frames=11, n_iter=30, classes=["N", "V", "S"])
valid_mask = [lbl in ["N", "V", "S"] for lbl in seg_100["aami_labels"]]
indices = np.where(valid_mask)[0]

train_seq = []
for idx in indices:
    train_seq.append(X_seq[idx*11 : (idx+1)*11])
train_X = np.vstack(train_seq)
train_lengths = [11] * len(indices)
train_labels = [seg_100["aami_labels"][idx] for idx in indices]

hmm.fit(train_X, train_lengths, train_labels, balanced_priors=True)
preds = hmm.predict(train_X, train_lengths)

metrics = compute_aami_metrics(train_labels, preds, classes=["N", "V", "S"])
print("Overall Accuracy:", metrics["overall_accuracy_pct"], "%")
print(format_metrics_table(metrics))

## 6. Viterbi Hidden State Decoding Visualization

In [ ]:
sample_beat = seg_100["segments"][10]
sample_obs = X_seq[10*11 : 11*11]
_, viterbi_states = hmm.decode_hidden_states(sample_obs, class_name="N")

fig = plot_hmm_viterbi_states(sample_beat, viterbi_states, class_name="N")
plt.show()